[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# Gestión de imágenes.

El concepto de "contenedor" implica que éstos pueden ser creados con el menor esfuerzo a partir de imágenes disponibles en repositorios locales y en línea. 

**ADVERTENCIA:**

Se da por sentado de que las celdas de los capítulos previos fueron ejecutadas y que existen imágenes cargadas en el sistema.

## Los *registry* y *Docker Hub*.

Una de las funcionalidades más relevantes de Docker es la de poder acceder a determinados repositorios en línea y traer imágenes o *Dockerfiles* para crear contenedores de forma local. A estos concentradores de repositorios se les conoce como *registry*.

### *Docker Hub*.

El *registry* al que Docker accede de forma predeterminada es *Docker Hub*.

*Docker Hub* cuenta con un amplio catálogo de repositorios públicos y permite a cualquier persona darse de alta y crear repositorios propios tanto públicos como privados dependiendo del plan de suscripción. El plan gratuito sólo permite crear sólo un repositorio privado.


*Docker Hub* se encuentra en https://hub.docker.com/

Cada vez que se utiliza ```docker run``` el daemon busca la imagen a la que se hace referencia en el sistema local y en caso de no encontrarla, la busca en Docker Hub.

### Identificación de una imagen o un *Dockerfile* en un *registry* mediante etiquetas.

Cada repositorio de un *registry* tiene un nombre único y a su vez cada imagen dentro del repositorio tienen un nombre y una etiqueta o *tag*.

El *tag* que se busca y asigna por defecto es ```latest```.

Las imágenes de docker pueden ser identificadas en un *registry* usando la siguiente estructura:

```
<nombre del repositorio>/<nombre>:<tag>
```

En el caso de acceder a *Docker Hub* sólo es necesario indicar el *tag*. En caso contrario es necesario indicar la *URL* del servidor del *registry*.

```
<URL>/<nombre del repositorio>/<nombre>:<tag>
```

**Ejemplo:**

* El repositorio oficial de *CentOS* es https://hub.docker.com/_/centos y en este repositorio se encuentran varios *Dockerfiles* e imágenes de diversas versiones de CentOS.
* La estructura que identifica al *Dockerfile* para construir una imagen de *CentOS 6.10* es: ```centos:6.10```

## El comando ```docker search```.

El comando ```docker search``` permite realizar una búsqueda de imágenes en los repositorios de *Docker Hub* a partir de un término.

```
docker search <argumentos> <termino>
```

Referencia: https://docs.docker.com/engine/reference/commandline/search/

**Ejemplo:** 

Se realizará una búsqueda de todas las imágines y Dockerfiles que coincidan con el término *cloudevel*.

In [ ]:
!docker search cloudevel

**ADVERTENCIA:** Existe una enorme cantidad de recursos disponibles en *Docker Hub*, pero no todos estos recursos son seguros y sólo una pequeña parte de estos ha sido validado por Docker o por los fabricantes. Se recomienda utilizar únicamente recursos oficiales o de plena confianza.

## El comando ```docker login``` (opcional).

Es muy importante que los *registry* cuenten con métodos de autenticación que garanticen la seguridad de los usuarios, por lo que para aceder a *Docker Hub* y a *registries* que hayan implementado dichos métodos es necesario presentar las credenciales adecuadas.

El comando ```docker login``` permite la autenticación de un usuario a un *registry*.

```docker login <argumentos>```

Hay muchas formas de autenticarse mediante ```docker login``` y se recomienda consultar la siguiente referncia:

https://docs.docker.com/engine/reference/commandline/login/

En caso de que el registro sea exitoso, se creará una sesión en el equipo local que permitiría interacrtuar con el *registry*.

El comando ```docker login``` inicia una sesión en un *registry*. Para *Docker Hub*, se recomienda un *token* de acceso personal (*personal access token*) en lugar de la contraseña de la cuenta. Se crea en https://app.docker.com/settings/personal-access-tokens.

* El parámetro ```--password-stdin``` lee la contraseña o el *token* desde la entrada estándar, de modo que no queda en el historial del *shell* ni en la lista de procesos.
* El usuario y el *token* **no** se escriben en el cuaderno: se leen de las variables de entorno ```DOCKERHUB_USER``` y ```DOCKERHUB_TOKEN```.

Desde una terminal:

```bash
echo "$DOCKERHUB_TOKEN" | docker login -u "$DOCKERHUB_USER" --password-stdin
```

**Cómo definir las variables:**

* En *GitHub Codespaces*: en la configuración de su cuenta de *GitHub*, apartado *Codespaces*, cree los secretos ```DOCKERHUB_USER``` y ```DOCKERHUB_TOKEN``` y autorícelos para este repositorio. Al crear o reiniciar el *Codespace*, llegan como variables de entorno.
* En un equipo propio: copie ```.env.example``` a ```.env``` y escriba ahí los valores. El archivo ```.env``` está en ```.gitignore``` y no se sube al repositorio.

**Advertencia:** ```docker login``` guarda el *token* en ```~/.docker/config.json```, codificado pero **no cifrado**, salvo que se configure un *credential helper*. Cierre la sesión con ```docker logout``` al terminar.

* La siguiente celda carga el archivo ```.env```, si existe, sin mostrar los valores, e indica si las variables están definidas. En *GitHub Codespaces* no hace falta el archivo.

In [ ]:
import os
from pathlib import Path

ENV = Path(".env")
if ENV.exists():
    for linea in ENV.read_text(encoding="utf-8").splitlines():
        linea = linea.strip()
        if linea and not linea.startswith("#") and "=" in linea:
            clave, valor = linea.split("=", 1)
            os.environ.setdefault(clave.strip(), valor.strip())

for variable in ("DOCKERHUB_USER", "DOCKERHUB_TOKEN"):
    print(f"{variable} definida: {bool(os.environ.get(variable))}")

* La siguiente celda inicia la sesión con esas variables. ```printenv``` entrega el *token* directamente a ```docker login``` por la entrada estándar.

In [ ]:
!printenv DOCKERHUB_TOKEN | docker login --username "$(printenv DOCKERHUB_USER)" --password-stdin

## El comando ```docker image```.

Este comando cuenta a su vez con subcomandos especializados para la gestión de imágenes.

### El comando ```docker image ls```.

Este comando es idéntico a ```docker images``` y despliega un listado de las imágenes existentes en un entorno local.

```
docker image ls
``` 

Referencia: https://docs.docker.com/engine/reference/commandline/image_ls/

**Ejemplo:**

* La siguiente celda desplegará el listado de imágenes disponibles.

In [ ]:
!docker image ls

In [ ]:
!docker images

In [ ]:
!docker image ls nginx

### El comando ```docker image build```.

Este comando es idéntico a ```docker build``` y permite crear una imagen a partir de un *Dockerfile*.

```
docker image build <dockerfile>
```

Referencia: https://docs.docker.com/engine/reference/commandline/image_build/

En el capítulo [04_dockerfile.ipynb](04_dockerfile.ipynb) se exploró la forma de crear imágenes a partir de un *Dockerfile*.

### El comando ```docker image pull```.

Este comando es idéntico a ```docker pull``` y permite acceder a un *registry* y descargar o construir una imagen a partir de un repositorio.

```
docker image pull <fuente de la imagen>
```

Referencia: https://docs.docker.com/engine/reference/commandline/image_pull/

**Ejemplos:**


* Se construirá la imagen oficial de la versión ```6.10``` de CentOS desde *Docker Hub*.

In [ ]:
!docker image pull centos:6.10

In [ ]:
!docker image ls --filter "reference=centos"

* Se descargará la imagen *cloudevel/cd331:version_1* localizada en https://hub.docker.com/r/cloudevel/cd331/tags

In [ ]:
!docker image pull cloudevel/cd331:latest-1

In [ ]:
!docker images

### El comando ```docker image inspect```.

Este comando regresa un documento *JSON* con los detalles de una imagen.

```
docker image inspect <imagen>
```

Referencia: https://docs.docker.com/engine/reference/commandline/image_inspect/

**Ejemplo:**

* Se desplegarán los detalles de la imagen ```centos:6.10```.

In [ ]:
!docker image inspect centos:6.10

### El comando ```docker image history```.

Este comando despliega la historia de modificaciones de una imagen.

```
docker image history <imagen>
```


Referencia: https://docs.docker.com/engine/reference/commandline/image_history/

**Ejemplo**

* Se desplegarán la historia de la imagen ```cloudevel/cd331:version_1```.

In [ ]:
!docker image history nginx_image

### El comando ```docker image rm```.

Este comando es idéntico a ```docker rmi``` y elimina la imagen de un entorno local. 


```
docker image rm <imagenes>
```

Es importante hacer notar que no es posible eliminar una imagen si existen contenedores creados a partir de éstas.

Referencia: https://docs.docker.com/engine/reference/commandline/image_rm/

**Ejemplo:**

* Se creará y ejecutará un contenedor a partir de la imagen más reciente de la versión `20.04` del repositorio oficial de *Ubuntu*.

In [ ]:
!docker run -dit --name ubuntu-ejemplo ubuntu:20.04

In [ ]:
!docker images

* Se intentará eliminar la imagen ```ubuntu:20.04```, pero debido a que existe un contenedor construido a partir de dicha imagen, no será posible.

In [ ]:
!docker image rm ubuntu:20.04

* Se detendrá y eliminará al contenedor ```ubuntu-ejemplo```.
* Se eliminará la imagen ```ubuntu:20.04```.

In [ ]:
!docker stop ubuntu-ejemplo

In [ ]:
!docker rm ubuntu-ejemplo

In [ ]:
!docker image rm ubuntu:20.04

In [ ]:
!docker images 

### El comando ```docker image prune```.

Este comando eliminará todas aquellas imágenes que no estén siendo utilizadas.

```
docker image prune
```

Referencia: https://docs.docker.com/engine/reference/commandline/image_prune/

**Ejemplo:**

* Se eliminarán todas las imágenes que no estén siendo utilzadas. 

In [ ]:
!docker run -dit --name ubuntu-exp ubuntu

In [ ]:
!docker images

In [ ]:
!docker image prune -af

In [ ]:
!docker images

### El comando ```docker image import```.

Este comando es idéntico a ```docker import``` y permite importar el contenido de un archivo con extensión ```.tar```, desde el cual se puede construir una imagen.

```
docker image import <archivo> <nombre>:<tag>
```

Referencia: https://docs.docker.com/engine/reference/commandline/image_import/

In [ ]:
!docker images

In [ ]:
!docker image import nginx.tar servidor:latest

In [ ]:
!docker image ls servidor

### El comando ```docker image load```.

Este comando es idéntico a ```docker load``` y es similar a ```docker image import```, pero se puede escoger entre un archivo tar o STDIN.

Referencia: https://docs.docker.com/engine/reference/commandline/image_load/

### El comando ```docker image save```.

Este comando es idéntico a ```docker save``` y puede guardar una o varias imágenes en un archivo tar o STDOUT.

Referencia: https://docs.docker.com/engine/reference/commandline/image_save/

In [ ]:
!docker image save nginx_image -o ultimo.tar

In [ ]:
!ls ultimo.tar

### El comando ```docker image tag```.

Este comando es idéntico a ```docker tag``` y se utiliza para asignarle nombre y tag a una imagen.

Referencia: https://docs.docker.com/engine/reference/commandline/image_tag/

In [ ]:
!docker images

In [ ]:
!docker tag nginx_image cloudevel/nginx:latest

In [ ]:
!docker image list

### El comando ```docker image push```.

Este comando se utiliza para subir una imagen a un regsistry.

Referencia: https://docs.docker.com/engine/reference/commandline/image_push/

**Ejemplo opcional (requiere una cuenta propia de *Docker Hub*):**

* Requiere haber iniciado sesión con ```docker login```, como se explicó antes.
* La siguiente celda etiqueta la imagen con el nombre de usuario de *Docker Hub* (tomado de la variable ```DOCKERHUB_USER```) y la sube a un repositorio de esa cuenta.
* La celda posterior cierra la sesión con ```docker logout```, que borra el *token* guardado en ```~/.docker/config.json```.

In [ ]:
!docker tag cloudevel/nginx:latest "$(printenv DOCKERHUB_USER)/cd501-nginx:1"
!docker push "$(printenv DOCKERHUB_USER)/cd501-nginx:1"

In [ ]:
!docker logout

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2027.</p>